In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

In [2]:
import pandas as pd

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression

In [4]:
from sklearn.model_selection import train_test_split

In [5]:
from sklearn.metrics import (
    accuracy_score,
    classification_report
)

---

In [6]:
# Step 1: Create Dataset

data = pd.DataFrame({
    "Age": [
        22, 25, 30, 35, 40,
        45, 50, 55, 60, 65
    ],
    "BloodPressure": [
        110, 115, 120, 125, 130,
        145, 150, 160, 170, 180
    ],
    "SugarLevel": [
        80, 85, 90, 95, 100,
        130, 140, 150, 160, 180
    ],
    "Cholesterol": [
        150, 160, 170, 180, 190,
        220, 230, 240, 250, 260
    ],
    "HeartRate": [
        70, 72, 74, 75, 78,
        85, 88, 90, 92, 95
    ],
    "Admission": [
        0, 0, 0, 0, 0,
        1, 1, 1, 1, 1
    ]
})

In [7]:
data

,Age,BloodPressure,SugarLevel,Cholesterol,HeartRate,Admission
0,22,110,80,150,70,0
1,25,115,85,160,72,0
2,30,120,90,170,74,0
3,35,125,95,180,75,0
4,40,130,100,190,78,0
5,45,145,130,220,85,1
6,50,150,140,230,88,1
7,55,160,150,240,90,1
8,60,170,160,250,92,1
9,65,180,180,260,95,1


---

In [8]:
# Step 2: Select Features and Target

X = data[[
    "Age",
    "BloodPressure",
    "SugarLevel",
    "Cholesterol",
    "HeartRate"
]]

y = data["Admission"]

---

In [9]:
# Step 3: Standardize Data

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

---

In [10]:
# Step 4: Apply PCA

# Reduce 5 columns to 2 columns

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

print("\nData after PCA:\n")

print(X_pca)


Data after PCA:

[[-3.06281093e+00  2.04637651e-01]
 [-2.58182830e+00  1.45206541e-01]
 [-2.03733758e+00 -1.94632981e-03]
 [-1.54452660e+00 -1.50986425e-01]
 [-9.48356156e-01 -2.96252072e-01]
 [ 6.18196843e-01  8.74993933e-03]
 [ 1.28068113e+00 -3.45275096e-02]
 [ 1.98895293e+00 -3.07213241e-02]
 [ 2.69722473e+00 -2.69151386e-02]
 [ 3.58980394e+00  1.82754667e-01]]


---

In [11]:
# Step 5: Apply K-Means

kmeans = KMeans(n_clusters=2, random_state=42)

clusters = kmeans.fit_predict(X_pca)

In [12]:
# Add cluster column

data["Cluster"] = clusters

print("\nClusters:\n")

print(data[["Age", "BloodPressure", "Cluster"]])


Clusters:

   Age  BloodPressure  Cluster
0   22            110        0
1   25            115        0
2   30            120        0
3   35            125        0
4   40            130        0
5   45            145        1
6   50            150        1
7   55            160        1
8   60            170        1
9   65            180        1


---

In [13]:
# Step 6: Add Cluster to Features

X_final = pd.DataFrame(X_pca, columns=["PCA_1", "PCA_2"])

X_final["Cluster"] = clusters

---

In [14]:
# Step 7: Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)

---

In [15]:
# Step 8: Train Logistic Regression

model = LogisticRegression()

model = model.fit(X_train, y_train)

---

In [16]:
# Step 9: Prediction

y_pred = model.predict(X_test)

---

In [17]:
# Step 10: Model Evaluation

accuracy = accuracy_score(y_test, y_pred)

print(f"\nAccuracy: {accuracy}\n")

print("\nClassification Report:\n")

print(classification_report(y_test, y_pred))


Accuracy: 1.0


Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



---

In [18]:
# Step 11: Predict New Patient

new_patient = pd.DataFrame({
    "Age": [58],
    "BloodPressure": [165],
    "SugarLevel": [145],
    "Cholesterol": [235],
    "HeartRate": [90]
})

In [19]:
new_patient

,Age,BloodPressure,SugarLevel,Cholesterol,HeartRate
0,58,165,145,235,90


In [20]:
# Standardize

new_scaled = scaler.transform(new_patient)

In [21]:
# PCA

new_pca = pca.transform(new_scaled)

In [22]:
# Find cluster

new_cluster = kmeans.predict(new_pca)

In [23]:
# Final input

new_data = pd.DataFrame(new_pca, columns=["PCA_1", "PCA_2"])

new_data["Cluster"] = new_cluster

prediction = model.predict(new_data)

print("\nPrediction:\n")

if prediction[0] == 1:
    print("Patient should be ADMITTED")
else:
    print("Patient can be DISCHARGED")


Prediction:

Patient should be ADMITTED
